<a href="https://colab.research.google.com/github/akhileshwar-pro/Text-Generation-with-GPT-2/blob/main/Text_Generation_with_GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch accelerate gradio huggingface_hub -q

print("All libraries installed successfully!")

All libraries installed successfully!


In [ ]:
text = """
Artificial Intelligence is transforming industries.

Machine learning helps computers learn patterns.

Python is widely used for AI development.

Deep learning models require large datasets.

Technology is improving healthcare and education.

Learning to code improves problem-solving skills.

Natural Language Processing helps computers understand text.

Cybersecurity is important in the digital world.

Cloud computing enables scalable applications.

Data science combines statistics and programming.
"""

In [ ]:
with open("dataset.txt", "w") as f:
    f.write(text)

print("Dataset file created successfully!")

Dataset file created successfully!


In [ ]:
with open("dataset.txt", "r") as f:
    print(f.read())


Artificial Intelligence is transforming industries.

Machine learning helps computers learn patterns.

Python is widely used for AI development.

Deep learning models require large datasets.

Technology is improving healthcare and education.

Learning to code improves problem-solving skills.

Natural Language Processing helps computers understand text.

Cybersecurity is important in the digital world.

Cloud computing enables scalable applications.

Data science combines statistics and programming.



In [ ]:
!ls

dataset.txt  sample_data


In [ ]:
!pip install transformers datasets torch accelerate gradio huggingface_hub -q

print("Libraries installed successfully!")

Libraries installed successfully!


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"

# Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Model
model = GPT2LMHeadModel.from_pretrained(model_name)

print("GPT-2 loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 loaded successfully!


In [ ]:
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling

# Load dataset
dataset = load_dataset("text", data_files={"train": "dataset.txt"})

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Dataset prepared successfully!")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset prepared successfully!


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"]
)

trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.374988
20,1.869463
30,1.498646


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=30, training_loss=2.247698974609375, metrics={'train_runtime': 39.6446, 'train_samples_per_second': 1.513, 'train_steps_per_second': 0.757, 'total_flos': 3919380480000.0, 'train_loss': 2.247698974609375, 'epoch': 3.0})

In [ ]:
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load saved model
model = GPT2LMHeadModel.from_pretrained("./gpt2-finetuned")

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-finetuned")

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

print("Fine-tuned model loaded!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Fine-tuned model loaded!


In [ ]:
def generate_text(prompt):

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True,
        top_k=40,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return generated_text

In [ ]:
print(generate_text("Artificial Intelligence"))

print()

print(generate_text("Python programming"))

print()

print(generate_text("Machine learning"))

Artificial Intelligence is transforming industries. Companies are developing tools that help automate tasks. Companies are using AI to automate tasks. Companies are using AI to automate tasks.

As the digital world changes, companies are learning about how to improve their processes. Companies are learning

Python programming is becoming increasingly accessible. And it's getting better. In the last few years, many companies have started developing software that translates programming languages into code. If you're interested in developing your own programming languages, read on to learn how to build a web

Machine learning systems (MLS) solve complex data problems. It enables researchers to understand complex data sets.

Machine learning models can learn patterns. In fact, it has been shown that machine learning models can learn patterns by learning algorithms. Learn more about computer


In [ ]:
import gradio as gr

# Chatbot function
def chatbot(prompt):
    return generate_text(prompt)

# Gradio interface
iface = gr.Interface(
    fn=chatbot,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Enter your prompt here...",
        label="Prompt"
    ),

    outputs=gr.Textbox(
        label="Generated Text"
    ),

    title="🚀 Custom GPT-2 AI Chatbot",

    description="Fine-tuned GPT-2 model using custom dataset"
)

iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be4f556382a858a031.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
iface.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be4f556382a858a031.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!zip -r gpt2-finetuned.zip gpt2-finetuned

  adding: gpt2-finetuned/ (stored 0%)
  adding: gpt2-finetuned/model.safetensors (deflated 7%)
  adding: gpt2-finetuned/generation_config.json (deflated 25%)
  adding: gpt2-finetuned/training_args.bin (deflated 53%)
  adding: gpt2-finetuned/config.json (deflated 52%)
  adding: gpt2-finetuned/checkpoint-20/ (stored 0%)
  adding: gpt2-finetuned/checkpoint-20/model.safetensors (deflated 7%)
  adding: gpt2-finetuned/checkpoint-20/generation_config.json (deflated 25%)
  adding: gpt2-finetuned/checkpoint-20/training_args.bin (deflated 53%)
  adding: gpt2-finetuned/checkpoint-20/config.json (deflated 52%)
  adding: gpt2-finetuned/checkpoint-20/rng_state.pth (deflated 26%)
  adding: gpt2-finetuned/checkpoint-20/trainer_state.json (deflated 58%)
  adding: gpt2-finetuned/checkpoint-20/scheduler.pt (deflated 61%)
  adding: gpt2-finetuned/checkpoint-20/tokenizer_config.json (deflated 48%)
  adding: gpt2-finetuned/checkpoint-20/tokenizer.json (deflated 82%)
  adding: gpt2-finetuned/checkpoint-20/op

In [ ]:
from google.colab import files

files.download("gpt2-finetuned.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Created dataset file at: .gradio/flagged/dataset1.csv
